In [6]:
!pip install PySastrawi

  Using cached PySastrawi-1.2.0-py2.py3-none-any.whl.metadata (892 bytes)
Using cached PySastrawi-1.2.0-py2.py3-none-any.whl (210 kB)



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# =========================================================
# IMPORT LIBRARY
# =========================================================
import sys
import os
sys.path.append(os.path.abspath(".."))
import pandas as pd
from supabase import create_client, Client

# Koneksi langsung
SUPABASE_URL = 'https://bnuzmrtiaciqlotxcgot.supabase.co'
SUPABASE_KEY = 'sb_publishable_Z8M8GISPVKMp1SGrQHrlLg_AZa8EOo-'
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

from src.preprocessing.clean_text import clean_text
from src.preprocessing.casefolding import casefolding
from src.preprocessing.tokenizing import tokenizing
from src.preprocessing.stopwords_id import get_stopwords
from src.preprocessing.stemming import stemming

print('✅ Semua import berhasil!')

# =========================================================
# LOAD DATA DARI SUPABASE (auto batch)
# =========================================================
print('📥 Mengambil data dari Supabase...')

all_data = []
batch_size = 1000
offset = 0

while True:
    response = supabase.table('scholar_articles') \
        .select('id, title, abstract, authors, year, source, category') \
        .range(offset, offset + batch_size - 1) \
        .execute()
    batch = response.data
    if not batch:
        break
    all_data.extend(batch)
    print(f'  → Batch {offset//batch_size + 1}: {len(batch)} data')
    if len(batch) < batch_size:
        break
    offset += batch_size

df = pd.DataFrame(all_data)
print(f'✅ Total data: {len(df)} baris')
df.head()


✅ Semua import berhasil!
📥 Mengambil data dari Supabase...
  → Batch 1: 200 data
✅ Total data: 200 baris


,id,title,abstract,authors,year,source,category
0,1,Machine learning for microbiologists,… how to evaluate a machine learning model and...,"F Asnicar, AM Thomas, A Passerini…",2024,Nature Reviews …,machine learning
1,2,International conference on machine learning,"In this paper, we make the key delineation on ...","W Li, C Wang, G Cheng, Q Song",2023,Transactions on machine learning …,machine learning
2,3,What is machine learning?,… that one can employ in machine learning (ML)...,J Bell,2022,Machine learning and the city: applications in …,machine learning
3,4,Amnesiac machine learning,… It gives EU residents the ability to request...,"L Graves, V Nagisetty, V Ganesh",2021,… of the AAAI conference on artificial …,machine learning
4,5,Designing nanotheranostics with machine learning,"… As a key branch of artificial intelligence, ...","L Rao, Y Yuan, X Shen, G Yu, X Chen",2024,Nature Nanotechnology,machine learning


In [13]:
# =========================================================
# LOAD DATA DARI SUPABASE (auto batch)
# =========================================================
print('📥 Mengambil data dari Supabase...')

all_data = []
batch_size = 1000
offset = 0

while True:
    response = supabase.table('scholar_articles') \
        .select('id, title, abstract, authors, year, source, category') \
        .range(offset, offset + batch_size - 1) \
        .execute()
    batch = response.data
    if not batch:
        break
    all_data.extend(batch)
    print(f'  → Batch {offset//batch_size + 1}: {len(batch)} data')
    if len(batch) < batch_size:
        break
    offset += batch_size

df = pd.DataFrame(all_data)
print(f'✅ Total data: {len(df)} baris')
df.head()

📥 Mengambil data dari Supabase...
  → Batch 1: 200 data
✅ Total data: 200 baris


,id,title,abstract,authors,year,source,category
0,1,Machine learning for microbiologists,… how to evaluate a machine learning model and...,"F Asnicar, AM Thomas, A Passerini…",2024,Nature Reviews …,machine learning
1,2,International conference on machine learning,"In this paper, we make the key delineation on ...","W Li, C Wang, G Cheng, Q Song",2023,Transactions on machine learning …,machine learning
2,3,What is machine learning?,… that one can employ in machine learning (ML)...,J Bell,2022,Machine learning and the city: applications in …,machine learning
3,4,Amnesiac machine learning,… It gives EU residents the ability to request...,"L Graves, V Nagisetty, V Ganesh",2021,… of the AAAI conference on artificial …,machine learning
4,5,Designing nanotheranostics with machine learning,"… As a key branch of artificial intelligence, ...","L Rao, Y Yuan, X Shen, G Yu, X Chen",2024,Nature Nanotechnology,machine learning


In [14]:
# =========================================================
# STEP 1 - GABUNGKAN TITLE + ABSTRACT
# =========================================================
df['full_text'] = df['title'].fillna('') + ' ' + df['abstract'].fillna('')
print('✅ Step 1 - Gabungkan title + abstract')
df[['title', 'full_text']].head(3)


✅ Step 1 - Gabungkan title + abstract


,title,full_text
0,Machine learning for microbiologists,Machine learning for microbiologists … how to ...
1,International conference on machine learning,International conference on machine learning I...
2,What is machine learning?,What is machine learning? … that one can emplo...


In [15]:
# =========================================================
# STEP 2 - TEXT CLEANSING
# =========================================================
df['cleaned'] = df['full_text'].apply(clean_text)
print('✅ Step 2 - Text Cleansing selesai')
df[['full_text', 'cleaned']].head(3)


✅ Step 2 - Text Cleansing selesai


,full_text,cleaned
0,Machine learning for microbiologists … how to ...,machine learning for microbiologists … how to ...
1,International conference on machine learning I...,international conference on machine learning i...
2,What is machine learning? … that one can emplo...,what is machine learning … that one can employ...


In [17]:
# =========================================================
# STEP 3 - CASE FOLDING
# =========================================================
df['casefolded'] = df['cleaned'].apply(casefolding)
print('✅ Step 3 - Case Folding selesai')
df[['cleaned', 'casefolded']].head(3) 



✅ Step 3 - Case Folding selesai


,cleaned,casefolded
0,machine learning for microbiologists … how to ...,machine learning for microbiologists … how to ...
1,international conference on machine learning i...,international conference on machine learning i...
2,what is machine learning … that one can employ...,what is machine learning … that one can employ...


In [18]:
# =========================================================
# STEP 4 - TOKENIZATION
# =========================================================
df['tokens'] = df['casefolded'].apply(tokenizing)
print('✅ Step 4 - Tokenization selesai')
df[['casefolded', 'tokens']].head(3)


✅ Step 4 - Tokenization selesai


,casefolded,tokens
0,machine learning for microbiologists … how to ...,"[machine, learning, for, microbiologists, …, h..."
1,international conference on machine learning i...,"[international, conference, on, machine, learn..."
2,what is machine learning … that one can employ...,"[what, is, machine, learning, …, that, one, ca..."


In [19]:
# =========================================================
# STEP 5 - STOPWORD REMOVAL
# =========================================================
stop_words = get_stopwords()

def remove_stopwords(tokens):
    if not isinstance(tokens, list):
        return []
    return [t for t in tokens if t not in stop_words and len(t) > 1]

df['tokens_clean'] = df['tokens'].apply(remove_stopwords)
print('✅ Step 5 - Stopword Removal selesai')
df[['tokens', 'tokens_clean']].head(3)


✅ Step 5 - Stopword Removal selesai


,tokens,tokens_clean
0,"[machine, learning, for, microbiologists, …, h...","[machine, learning, for, microbiologists, how,..."
1,"[international, conference, on, machine, learn...","[international, conference, on, machine, learn..."
2,"[what, is, machine, learning, …, that, one, ca...","[what, is, machine, learning, that, one, can, ..."


In [20]:
# =========================================================
# STEP 6 - STEMMING (Sastrawi - Nazief-Adriani)
# =========================================================
df['tokens_stemmed'] = df['tokens_clean'].apply(stemming)
print('✅ Step 6 - Stemming selesai')
df[['tokens_clean', 'tokens_stemmed']].head(3)

✅ Step 6 - Stemming selesai


,tokens_clean,tokens_stemmed
0,"[machine, learning, for, microbiologists, how,...","[machine, learning, for, microbiologists, how,..."
1,"[international, conference, on, machine, learn...","[international, conference, on, machine, learn..."
2,"[what, is, machine, learning, that, one, can, ...","[what, is, machine, learning, that, one, can, ..."


In [21]:
# =========================================================
# GABUNGKAN TOKEN → CLEANED_TEXT
# =========================================================
df['cleaned_text'] = df['tokens_stemmed'].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')

# Baru cek di sini ✅
print(f'Total baris diproses: {len(df)}')
print(f'Baris dengan cleaned_text kosong: {df["cleaned_text"].eq("").sum()}')

Total baris diproses: 200
Baris dengan cleaned_text kosong: 0


In [23]:
import os

# Naik satu level dari folder notebook/ ke paperCi/
base_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
save_path = os.path.join(base_dir, 'data', 'cleaned_papers.csv')

os.makedirs(os.path.join(base_dir, 'data'), exist_ok=True)
save_df = df[['id', 'title', 'abstract', 'authors', 'year', 'source', 'category', 'cleaned_text']].copy()
save_df.to_csv(save_path, index=False)

print(f'✅ Tersimpan: {save_path}')
print(f'📊 Total: {len(save_df)} baris')
save_df.head(5)

✅ Tersimpan: d:\Tugas Akhir\paperCi\data\cleaned_papers.csv
📊 Total: 200 baris


,id,title,abstract,authors,year,source,category,cleaned_text
0,1,Machine learning for microbiologists,… how to evaluate a machine learning model and...,"F Asnicar, AM Thomas, A Passerini…",2024,Nature Reviews …,machine learning,machine learning for microbiologists how to ev...
1,2,International conference on machine learning,"In this paper, we make the key delineation on ...","W Li, C Wang, G Cheng, Q Song",2023,Transactions on machine learning …,machine learning,international conference on machine learning i...
2,3,What is machine learning?,… that one can employ in machine learning (ML)...,J Bell,2022,Machine learning and the city: applications in …,machine learning,what is machine learning that one can employ i...
3,4,Amnesiac machine learning,… It gives EU residents the ability to request...,"L Graves, V Nagisetty, V Ganesh",2021,… of the AAAI conference on artificial …,machine learning,amnesiac machine learning it gives eu resident...
4,5,Designing nanotheranostics with machine learning,"… As a key branch of artificial intelligence, ...","L Rao, Y Yuan, X Shen, G Yu, X Chen",2024,Nature Nanotechnology,machine learning,designing nanotheranostics with machine learni...
